In [2]:
import cv2
import keras
import numpy as np
import matplotlib.pyplot as plt
import os
import tensorflow as tf
import time
from keras import backend as K
from keras import layers
# from datasets import load_dataset
from PIL import Image

In [13]:
image = np.load("./HSSDataset/HSS_VOC/npy_multichannel/1_frame_0001.npy").astype(np.float32)[:-1, :-1, :]
image.shape

(216, 408, 25)

In [ ]:
# Formatting the data inside particular directories based on train.txt, val.txt
# and test.txt files

import os
import shutil


dataset_root = "./HSSDataset/HSS_VOC"
hsi_data_dir = "npy_multichannel"
mask_data_dir = "Mask"
split_dir = "split"
destination_root = "./HSSDataset_organized"

os.makedirs(destination_root, exist_ok=True)

splits = {
    "train": "train.txt",
    "val": "val.txt",
    "test": "test.txt"
}

print("Starting dataset organization...")

for split_name, split_file in splits.items():
    split_filepath = os.path.join(dataset_root, split_dir, split_file)

    dest_image_dir = os.path.join(destination_root, split_name, "images")
    dest_mask_dir = os.path.join(destination_root, split_name, "masks")

    os.makedirs(dest_image_dir, exist_ok=True)
    os.makedirs(dest_mask_dir, exist_ok=True)

    print(f"Processing {split_name} split using {split_filepath}")

    with open(split_filepath, 'r') as f:
        filenames = f.readlines()

        for file_base in filenames:
            file_base = file_base.strip() # Remove newline characters

            # Source paths
            src_image_path = os.path.join(dataset_root, hsi_data_dir, file_base + '.npy')
            src_mask_path = os.path.join(dataset_root, mask_data_dir, file_base + '.png')

            # Destination paths
            dest_image_path = os.path.join(dest_image_dir, file_base + '.npy')
            dest_mask_path = os.path.join(dest_mask_dir, file_base + '.png')

            if os.path.exists(src_image_path) and os.path.exists(src_mask_path):
                # shutil.copy(src_image_path, dest_image_path)
                # Remove last row and column to match mask dimensions
                hsi_img = np.load(src_image_path).astype(np.float32)[:-1, :-1, :] 
                # print(f"Loaded HSI image from {src_image_path} with shape {hsi_img.shape}")
                np.save(dest_image_path, hsi_img)
                time.sleep(0.01)

                # shutil.copy(src_mask_path, dest_mask_path)
                raw_mask_content = tf.io.read_file(src_mask_path)
                mask_tensor = tf.image.decode_png(raw_mask_content, channels=1)

                # Resize mask using TensorFlow (target height, width)
                # The original mask resolution is 1088x2048, HSI is 217x409
                # tf.image.resize expects (height, width)
                # slice off the last row and column to match the HSI dimensions
                resized_mask_tensor = tf.image.resize(
                    tf.cast(mask_tensor, tf.float32), # Cast to float for resize
                    (217, 409), # Target (height, width)
                    method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
                )[:-1, :-1]

                # Cast back to uint8 for PNG encoding
                resized_mask_tensor = tf.cast(resized_mask_tensor, tf.uint8)
                # print(f"Resized mask from {mask_tensor.shape} to {resized_mask_tensor.shape} for {file_base}")

                # Write mask using TensorFlow
                encoded_mask = tf.image.encode_png(resized_mask_tensor)
                tf.io.write_file(dest_mask_path, encoded_mask)
                time.sleep(0.01)
            else:
                print(f"Warning: Skipping {file_base} - Image or mask not found: {src_image_path}, {src_mask_path}")

print("Dataset organization complete!")

Starting dataset organization...
Processing train split using ./HSSDataset/HSS_VOC\split\train.txt
Processing val split using ./HSSDataset/HSS_VOC\split\val.txt
Processing test split using ./HSSDataset/HSS_VOC\split\test.txt
Dataset organization complete!
